# The North Face e-commerce: NLP solution

This notebook answers the 3 requested parts:
1. product clustering,
2. recommender system,
3. topic modeling with LSA.

Before running it, place Kaggle's `sample-data.csv` next to this notebook.

## Setup

The dataset preview visible on Kaggle shows 2 columns: `id` and `description`, for 500 products.

In [ ]:
from pathlib import Path
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from wordcloud import WordCloud

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 120)

In [ ]:
data_path = Path('sample-data.csv')
if not data_path.exists():
    raise FileNotFoundError("Add sample-data.csv next to this notebook before running the analysis.")

corpus = pd.read_csv(data_path)
corpus.head()

## 1. Preprocessing

In [ ]:
def clean_text(text):
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = re.sub(r'[^A-Za-z\s-]', ' ', text)
    text = text.replace('-', ' ')
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

corpus['clean_description'] = corpus['description'].map(clean_text)
corpus[['id', 'clean_description']].head()

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, max_df=0.9)
X = vectorizer.fit_transform(corpus['clean_description'])
feature_names = vectorizer.get_feature_names_out()
print(X.shape)

## 2. Clustering

Recommended baseline, aligned with public notebooks built on the same dataset:
- DBSCAN
- cosine distance
- start around `eps=0.7` and `min_samples=3`

In [ ]:
def evaluate_dbscan(X, target_clusters=12):
    best_score = None
    best = {'eps': 0.7, 'min_samples': 3}
    best_labels = None
    for eps in [0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]:
        for min_samples in [2, 3, 4]:
            labels = DBSCAN(eps=eps, min_samples=min_samples, metric='cosine', algorithm='brute').fit_predict(X)
            clusters = [c for c in set(labels) if c != -1]
            if not clusters:
                continue
            sizes = Counter([c for c in labels if c != -1])
            noise = float(np.mean(labels == -1))
            score = -abs(len(clusters) - target_clusters) * 2 - noise * 8 - max(sizes.values()) / len(labels) * 3
            if best_score is None or score > best_score:
                best_score = score
                best = {'eps': eps, 'min_samples': min_samples}
                best_labels = labels
    return best, best_labels

best_params, labels = evaluate_dbscan(X)
corpus['cluster_id'] = labels
best_params

In [ ]:
X_df = pd.DataFrame(X.toarray(), columns=feature_names)

def top_terms(cluster_id, n=8):
    subset = X_df.loc[corpus['cluster_id'] == cluster_id]
    if subset.empty:
        return []
    return subset.mean(axis=0).sort_values(ascending=False).head(n).index.tolist()

cluster_summary = corpus['cluster_id'].value_counts().rename_axis('cluster_id').reset_index(name='n_items')
cluster_summary['top_terms'] = cluster_summary['cluster_id'].map(top_terms)
cluster_summary.sort_values('cluster_id')

In [ ]:
cluster_ids = [c for c in cluster_summary['cluster_id'].tolist() if c != -1][:6]
fig, axes = plt.subplots(len(cluster_ids), 1, figsize=(14, 4 * max(1, len(cluster_ids))))
axes = np.array(axes).reshape(-1)
for ax, cluster_id in zip(axes, cluster_ids):
    text = ' '.join(corpus.loc[corpus['cluster_id'] == cluster_id, 'clean_description'])
    cloud = WordCloud(width=900, height=300, background_color='white', colormap='YlGnBu').generate(text)
    ax.imshow(cloud, interpolation='bilinear')
    ax.set_title(f"Cluster {cluster_id} | {', '.join(top_terms(cluster_id, 5))}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Recommender system

In [ ]:
similarity_matrix = cosine_similarity(X)
id_to_index = {int(item_id): idx for idx, item_id in enumerate(corpus['id'])}

def find_similar_items(item_id, top_k=5):
    item_id = int(item_id)
    idx = id_to_index[item_id]
    cluster_id = corpus.iloc[idx]['cluster_id']
    candidates = []
    for other_idx, score in enumerate(similarity_matrix[idx]):
        if other_idx == idx or corpus.iloc[other_idx]['cluster_id'] != cluster_id:
            continue
        candidates.append((score, int(corpus.iloc[other_idx]['id'])))
    return [item_id for _, item_id in sorted(candidates, reverse=True)[:top_k]]


def show_recommendations(item_id):
    print(corpus.loc[corpus['id'] == int(item_id), 'description'].iloc[0])
    print('
Suggestions:')
    for candidate_id in find_similar_items(item_id):
        print(f"- #{candidate_id}: {corpus.loc[corpus['id'] == candidate_id, 'description'].iloc[0]}")

In [ ]:
# Optional demo cell
# product_id = int(input('Enter a product id: '))
# show_recommendations(product_id)

## 4. Topic modeling with LSA

In [ ]:
svd_model = TruncatedSVD(n_components=12, random_state=42)
topic_matrix = svd_model.fit_transform(X)
topic_encoded_df = pd.DataFrame(topic_matrix, columns=[f'topic_{i}' for i in range(topic_matrix.shape[1])])
topic_encoded_df['main_topic'] = topic_encoded_df.abs().idxmax(axis=1)
topic_encoded_df.head()

In [ ]:
topic_terms = pd.DataFrame(np.abs(svd_model.components_), columns=feature_names, index=[f'topic_{i}' for i in range(svd_model.components_.shape[0])])
topic_summary = topic_encoded_df['main_topic'].value_counts().rename_axis('main_topic').reset_index(name='n_items')
topic_summary['top_terms'] = topic_summary['main_topic'].map(lambda topic: topic_terms.loc[topic].sort_values(ascending=False).head(8).index.tolist())
topic_summary

In [ ]:
selected_topics = topic_summary['main_topic'].tolist()[:6]
fig, axes = plt.subplots(len(selected_topics), 1, figsize=(14, 4 * max(1, len(selected_topics))))
axes = np.array(axes).reshape(-1)
for ax, topic in zip(axes, selected_topics):
    text = ' '.join(corpus.loc[topic_encoded_df['main_topic'] == topic, 'clean_description'])
    cloud = WordCloud(width=900, height=300, background_color='white', colormap='cividis').generate(text)
    ax.imshow(cloud, interpolation='bilinear')
    ax.set_title(f"{topic} | {', '.join(topic_summary.loc[topic_summary['main_topic'] == topic, 'top_terms'].iloc[0][:5])}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## Final conclusion

- DBSCAN gives product neighborhoods that can directly feed a content-based recommender.
- LSA helps challenge the existing catalog taxonomy.
- Next improvement: add spaCy lemmatization and re-run on the full 500-row CSV before the oral defense.